# DINOTxt Experiments — Semantic Heatmap
This Notebook contains different experiments done with DINOtxt to find the best results for various queries. The notebook contains code for both an indoor and outdoor dataset, on which we majorly focus on the indoor. The final part of the code gives a video of the whole scene and the queried word highlighted clearly in the scene.

## 1. Configuration

In [ ]:
import os

PROJECT_ROOT = os.path.expanduser("~/Universal-Semantic-World-Models")
SCENE_TYPE   = "indoor"   # "indoor" | "outdoor"

if SCENE_TYPE == "indoor":
    DATASET_IMAGES_DIR = os.path.join(PROJECT_ROOT, "dataset/replica/room2", "results")
    RESIZED_IMAGES_DIR = os.path.join(PROJECT_ROOT, "dataset/replica/room2", "resized_images_indoor")
    OUTPUT_DIR         = os.path.join(PROJECT_ROOT, "output",  "dinotxt_heatmaps_indoor")
elif SCENE_TYPE == "outdoor":
    DATASET_IMAGES_DIR = os.path.join(PROJECT_ROOT, "dataset/tanksandtemples", "images")
    RESIZED_IMAGES_DIR = os.path.join(PROJECT_ROOT, "dataset/tanksandtemples", "resized_images_outdoor")
    OUTPUT_DIR         = os.path.join(PROJECT_ROOT, "output",  "dinotxt_heatmaps_outdoor")
else:
    raise ValueError(f"SCENE_TYPE must be 'indoor' or 'outdoor', got: {SCENE_TYPE!r}")

DINOV3_REPO      = os.path.join(PROJECT_ROOT, "dinov3")
PRETRAINED_DIR   = os.path.join(PROJECT_ROOT, "pretrainedmodels")
DINOTXT_WEIGHTS  = os.path.join(PRETRAINED_DIR, "dinov3_vitl16_dinotxt_vision_head_and_text_encoder-a442d8f5.pth")
BACKBONE_WEIGHTS = os.path.join(PRETRAINED_DIR, "dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth")

print(f"Scene    : {SCENE_TYPE}")
print(f"Images   : {DATASET_IMAGES_DIR}")
print(f"Resized  : {RESIZED_IMAGES_DIR}")
print(f"Output   : {OUTPUT_DIR}\n")

for label, path in [("DINOTxt weights", DINOTXT_WEIGHTS), ("Backbone weights", BACKBONE_WEIGHTS)]:
    status = "✓" if os.path.exists(path) else "✗ NOT FOUND"
    print(f"{status}  {label}: {os.path.basename(path)}")


## 2. Device Setup

In [ ]:
import sys
import torch

if DINOV3_REPO not in sys.path:
    sys.path.insert(0, DINOV3_REPO)
if os.path.dirname(DINOV3_REPO) not in sys.path:
    sys.path.insert(0, os.path.dirname(DINOV3_REPO))

if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

AUTOCAST_DEVICE = "cpu" if DEVICE.type in ("mps", "cpu") else "cuda"

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}  |  autocast context: {AUTOCAST_DEVICE}")


## 3. Model Loading

In [ ]:
import torch

model, tokenizer = torch.hub.load(
    DINOV3_REPO,
    "dinov3_vitl16_dinotxt_tet1280d20h24l",
    source="local",
    weights=DINOTXT_WEIGHTS,
    backbone_weights=BACKBONE_WEIGHTS,
    pretrained=True,
)
model = model.eval().to(DEVICE)
print(f"Loaded  ({sum(p.numel() for p in model.parameters())/1e6:.0f}M parameters)  ->  {DEVICE}")


In [ ]:
import contextlib

def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast("cuda", dtype=torch.float16)
    return contextlib.nullcontext()


## 4. Image Preprocessing

In [ ]:
import numpy as np
from PIL import Image
import torchvision.transforms as T
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TARGET_W, TARGET_H = 1024, 576
PATCH_SIZE  = 16
N_PATCHES_W = TARGET_W // PATCH_SIZE
N_PATCHES_H = TARGET_H // PATCH_SIZE

IMAGE_TRANSFORM = T.Compose([
    T.Resize((TARGET_H, TARGET_W)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

def load_image(path_or_pil):
    img = Image.open(path_or_pil).convert("RGB") if isinstance(path_or_pil, str) else path_or_pil.convert("RGB")
    return IMAGE_TRANSFORM(img).unsqueeze(0).to(DEVICE)

def unnormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

print(f"Target resolution : {TARGET_W}x{TARGET_H}  |  patch grid : {N_PATCHES_W}x{N_PATCHES_H} = {N_PATCHES_W*N_PATCHES_H} patches")


## 5. Zero-Shot Classification Demo

In [ ]:
import urllib.request

EXAMPLE_IMAGE_URL = "https://dl.fbaipublicfiles.com/dinov2/images/example.jpg"

try:
    with urllib.request.urlopen(EXAMPLE_IMAGE_URL, timeout=10) as f:
        img_pil = Image.open(f).convert("RGB")
    print(f"Downloaded example image: {img_pil.size}")
except Exception as e:
    print(f"URL unavailable ({e}) — using first dataset image.")
    candidates = sorted(f for f in os.listdir(DATASET_IMAGES_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    img_pil = Image.open(os.path.join(DATASET_IMAGES_DIR, candidates[0])).convert("RGB")

plt.figure(figsize=(6, 4)); plt.imshow(img_pil); plt.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
texts       = ["photo of dogs", "photo of a chair", "photo of a bowl", "photo of a wooden floor"]
class_names = ["dog", "chair", "bowl", "wooden floor"]

image_tensor = load_image(img_pil)
tokenized    = tokenizer.tokenize(texts).to(DEVICE)

with autocast_ctx(), torch.no_grad():
    img_feat  = model.encode_image(image_tensor)
    text_feat = model.encode_text(tokenized)

img_feat  = img_feat  / img_feat.norm(dim=-1, keepdim=True)
text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
scores    = text_feat.cpu().float().numpy() @ img_feat.cpu().float().numpy().T

print("Zero-shot classification (cosine similarity):")
for name, s in sorted(zip(class_names, scores[:, 0]), key=lambda x: -x[1]):
    print(f"  {name:<14} {s:+.4f}  {'█' * int(s * 40 + 20)}")
print(f"\n-> Predicted: '{class_names[int(scores[:, 0].argmax())]}'")


## 6. Heatmap Pipeline

In [ ]:
import math
import cv2
from sklearn.cluster import MiniBatchKMeans

PROMPT_TEMPLATES_INDOOR = (
    "a photo of a {0}.", "a photo of the {0}.", "a photo of one {0}.",
    "a close-up photo of a {0}.", "a close-up photo of the {0}.",
    "a good photo of a {0}.", "a good photo of the {0}.",
    "a bright photo of a {0}.", "a bright photo of the {0}.",
    "a dark photo of a {0}.", "a dark photo of the {0}.",
    "a photo of a large {0}.", "a photo of a small {0}.",
    "a photo of the large {0}.", "a photo of the small {0}.",
    "a cropped photo of the {0}.", "a blurry photo of a {0}.",
    "a rendering of a {0}.", "a rendering of the {0}.",
    "a 3D render of a {0}.", "a 3D render of the {0}.",
    "a synthetic image of a {0}.", "a photorealistic render of a {0}.",
    "a computer-generated image of a {0}.",
    "a {0} inside a room.", "a {0} in an indoor scene.",
    "a {0} in a living room.", "a {0} in a furnished room.",
    "a {0} photographed indoors.", "a {0} under indoor lighting.",
    "a {0} seen from inside a building.", "a {0} against a wall.",
    "a {0} near a wall.", "a {0} on the floor.",
    "a {0} in the corner of a room.", "a {0} in a home interior.",
    "a photo of a {0} in an apartment.", "a {0} in a minimalist interior.",
    "a {0} with natural light coming through a window.",
    "a {0} in a well-lit room.", "a {0} casting a shadow on the floor.",
    "an interior design photo of a {0}.", "an architectural photo of a {0}.",
    "a photo of the clean {0}.", "a photo of the wooden {0}.",
    "a photo of the textured {0}.", "a photo of a hard to see {0}.",
    "a low resolution photo of the {0}.",
    "a frontal view of the {0}.", "a side view of the {0}.",
    "a top-down view of the {0}.", "a wide-angle photo of a {0}.",
    "a fish-eye view of a {0}.", "the {0} seen from above.",
    "the {0} seen from the side.", "a panoramic view including a {0}.",
)

PROMPT_TEMPLATES_OUTDOOR = (
    "a photo of a {0}.", "a photo of the {0}.", "a photo of one {0}.",
    "a close-up photo of a {0}.", "a close-up photo of the {0}.",
    "a good photo of a {0}.", "a good photo of the {0}.",
    "a bright photo of a {0}.", "a bright photo of the {0}.",
    "a dark photo of a {0}.", "a dark photo of the {0}.",
    "a photo of a large {0}.", "a photo of a small {0}.",
    "a photo of the large {0}.", "a photo of the small {0}.",
    "a cropped photo of the {0}.", "a blurry photo of a {0}.",
    "a photo of a {0} outdoors.", "a photo of a {0} in a plaza.",
    "a photo of a {0} in a courtyard.", "a photo of a {0} in a public space.",
    "a photo of a {0} in sunlight.", "a photo of a {0} on a sunny day.",
    "a photo of a {0} in the shade.", "a photo of a {0} in natural light.",
    "a photo of a {0} in front of a building.", "a photo of a {0} near a building.",
    "a photo of a {0} in an open area.", "a photo of a {0} in an urban setting.",
    "a {0} photographed outdoors.", "a {0} in a campus setting.",
    "a {0} on a brick paved path.",
    "a photo of a {0} on a pedestal.", "a photo of a {0} on a stone base.",
    "a photo of a {0} made of bronze.", "a photo of a bronze {0}.",
    "a photo of an outdoor {0}.", "a memorial {0} in a public square.",
    "a {0} sculpture in a plaza.", "a {0} monument outdoors.",
    "a {0} surrounded by greenery.", "a {0} next to trees.",
    "a {0} on paved ground.", "a {0} on brick pavement.",
    "a frontal view of the {0}.", "a side view of the {0}.",
    "a wide-angle photo of a {0}.", "the {0} seen from a low angle.",
    "the {0} seen from the side.", "a panoramic view including a {0}.",
    "a tourist photo of a {0}.",
    "a photo of the clean {0}.", "a photo of the weathered {0}.",
    "a photo of the textured {0}.", "a low resolution photo of the {0}.",
)

PROMPT_TEMPLATES = PROMPT_TEMPLATES_INDOOR if SCENE_TYPE == "indoor" else PROMPT_TEMPLATES_OUTDOOR
print(f"Templates : {SCENE_TYPE.upper()}  ({len(PROMPT_TEMPLATES)} templates)")


def build_text_features(class_names):
    feats = []
    for name in class_names:
        prompts = [t.format(name) for t in PROMPT_TEMPLATES]
        tokens  = tokenizer.tokenize(prompts).to(DEVICE)
        with autocast_ctx(), torch.no_grad():
            f = model.encode_text(tokens)
        f = f[:, 1024:]
        f = F.normalize(f.float(), p=2, dim=-1)
        f = F.normalize(f.mean(dim=0), p=2, dim=-1)
        feats.append(f)
        print(f"  '{name}'")
    return torch.stack(feats)


def encode_image_patches(img_tensor_3d):
    img = img_tensor_3d.unsqueeze(0)
    H, W = img.shape[-2:]
    new_H, new_W = math.ceil(H / 16) * 16, math.ceil(W / 16) * 16
    if (H, W) != (new_H, new_W):
        img = F.interpolate(img, size=(new_H, new_W), mode="bicubic", align_corners=False)
    with autocast_ctx(), torch.no_grad():
        _, patch_tokens, _ = model.encode_image_with_patch_tokens(img)
    patches = patch_tokens[0].float()
    patches = patches - 0.3 * patches.mean(dim=0, keepdim=True)
    return patches.reshape(new_H // 16, new_W // 16, -1)


def predict_whole_img(img_tensor_3d, text_features, global_mean=None):
    img = img_tensor_3d.unsqueeze(0)
    H, W = img.shape[-2:]
    new_H, new_W = math.ceil(H / 16) * 16, math.ceil(W / 16) * 16
    if (H, W) != (new_H, new_W):
        img = F.interpolate(img, size=(new_H, new_W), mode="bicubic", align_corners=False)
    with autocast_ctx(), torch.no_grad():
        _, patch_tokens, _ = model.encode_image_with_patch_tokens(img)
    patches = patch_tokens[0].float()
    mean    = global_mean if global_mean is not None else patches.mean(dim=0, keepdim=True)
    patches = patches - 0.3 * mean
    patches = patches.reshape(new_H // 16, new_W // 16, -1)
    patches = F.normalize(patches, p=2, dim=-1)
    try:
        scale = model.logit_scale.exp().float().clamp(max=100.0)
    except AttributeError:
        scale = torch.tensor(20.0)
    return torch.einsum("cd,hwd->chw", text_features, patches) * scale


def predict_slide_window(img_tensor_3d, text_features, side=512, stride=128):
    _, H, W   = img_tensor_3d.shape
    n_classes = text_features.shape[0]
    probs  = torch.zeros([n_classes, H, W], device=DEVICE)
    counts = torch.zeros([H, W],            device=DEVICE)
    h_grids   = max(H - side + stride - 1, 0) // stride + 1
    w_grids   = max(W - side + stride - 1, 0) // stride + 1
    n_windows = h_grids * w_grids
    done = 0
    for i in range(h_grids):
        for j in range(w_grids):
            y1 = i * stride; x1 = j * stride
            y2 = min(y1 + side, H); x2 = min(x1 + side, W)
            y1 = max(y2 - side, 0); x1 = max(x2 - side, 0)
            cos = predict_whole_img(img_tensor_3d[:, y1:y2, x1:x2], text_features)
            cos = F.interpolate(cos.unsqueeze(0), size=(y2 - y1, x2 - x1),
                                mode="bilinear", align_corners=False).squeeze(0)
            probs[:, y1:y2, x1:x2] += cos.softmax(dim=0)
            counts[y1:y2, x1:x2]   += 1
            done += 1
            print(f"  Window {done}/{n_windows}  [{y1}:{y2}, {x1}:{x2}]", end="\r")
    print()
    return probs / counts


def get_heatmaps_sliding(image_path_or_pil, text_features, side=512, stride=128, margin_threshold=0.08):
    from pathlib import Path
    img_pil    = Image.open(image_path_or_pil).convert("RGB") if isinstance(image_path_or_pil, (str, Path)) else image_path_or_pil.convert("RGB")
    img_tensor = load_image(img_pil).squeeze(0)
    probs      = predict_slide_window(img_tensor, text_features, side=side, stride=stride)
    raw_cos    = predict_whole_img(img_tensor, text_features)
    raw_cos    = F.interpolate(raw_cos.unsqueeze(0), size=(TARGET_H, TARGET_W),
                               mode="bicubic", align_corners=False).squeeze(0)
    pred_idx        = probs.argmax(dim=0).detach().cpu().numpy()
    top2            = torch.topk(probs, k=2, dim=0).values
    margin          = (top2[0] - top2[1]).detach().cpu().numpy()
    uncertain_mask  = margin < margin_threshold
    pred_idx_masked = pred_idx.copy()
    pred_idx_masked[uncertain_mask] = -1
    return raw_cos.detach().cpu().numpy(), pred_idx, pred_idx_masked, uncertain_mask


def get_early_patch_features(img_tensor, layer=8):
    import torch.nn as nn
    backbone = model.visual_model

    def find_blocks(module):
        for _, child in module.named_children():
            if isinstance(child, nn.ModuleList) and len(child) > 4:
                if any(hasattr(child[0], a) for a in ("attn", "norm1", "norm", "attention")):
                    return child
            result = find_blocks(child)
            if result is not None:
                return result
        return None

    blocks = find_blocks(backbone)
    if blocks is None:
        print("WARNING: transformer blocks not found — falling back to final-layer features.")
        with autocast_ctx(), torch.no_grad():
            _, _, pt = backbone.get_class_and_patch_tokens(img_tensor)
        return pt[:, -(N_PATCHES_H * N_PATCHES_W):, :]

    layer    = min(layer, len(blocks) - 1)
    captured = {}
    handle   = blocks[layer].register_forward_hook(
        lambda m, i, o: captured.__setitem__("x", o[0] if isinstance(o, tuple) else o)
    )
    with autocast_ctx(), torch.no_grad():
        _ = backbone(img_tensor)
    handle.remove()

    tokens = captured["x"]
    return tokens[:, -(N_PATCHES_H * N_PATCHES_W):, :]


def bilateral_filter_heatmaps(heatmaps_np, img_np, d=9, sigma_color=0.08, sigma_space=12):
    filtered = np.zeros_like(heatmaps_np)
    for i, h in enumerate(heatmaps_np):
        lo, hi      = h.min(), h.max()
        h_u8        = ((h - lo) / (hi - lo + 1e-8) * 255).astype(np.float32)
        h_f         = cv2.bilateralFilter(h_u8, d=d, sigmaColor=sigma_color * 255, sigmaSpace=sigma_space)
        filtered[i] = h_f / 255.0 * (hi - lo) + lo
    return filtered


def kmeans_refine(heatmaps_np, pixel_features_np, n_clusters=48):
    N, H, W = heatmaps_np.shape
    D       = pixel_features_np.shape[-1]
    flat    = pixel_features_np.reshape(-1, D).astype(np.float32)
    n_fit   = min(flat.shape[0], 20000)
    idx_fit = np.random.choice(flat.shape[0], n_fit, replace=False)
    km      = MiniBatchKMeans(n_clusters=n_clusters, batch_size=4096, n_init=5, random_state=0).fit(flat[idx_fit])
    labels  = km.predict(flat).reshape(H, W)
    heat_flat = heatmaps_np.reshape(N, -1)
    pred      = np.zeros((H, W), dtype=np.int32)
    for k in range(n_clusters):
        mask = labels.reshape(-1) == k
        if mask.any():
            pred[labels == k] = int(heat_flat[:, mask].mean(axis=1).argmax())
    return pred, labels


def get_heatmaps_full(image_path_or_pil, text_features,
                      sides=(512,), stride=256,
                      early_layer=8,
                      bilateral_d=9, bilateral_sc=0.08, bilateral_ss=12,
                      margin_threshold=0.08,
                      use_kmeans=True, n_clusters=48):
    from pathlib import Path
    img_pil    = Image.open(image_path_or_pil).convert("RGB") if isinstance(image_path_or_pil, (str, Path)) else image_path_or_pil.convert("RGB")
    img_tensor = load_image(img_pil).squeeze(0)
    img_np     = np.array(img_pil.resize((TARGET_W, TARGET_H), Image.LANCZOS)) / 255.0

    n_classes = text_features.shape[0]
    probs     = torch.zeros([n_classes, TARGET_H, TARGET_W], device=DEVICE)
    counts    = torch.zeros([TARGET_H, TARGET_W],            device=DEVICE)
    total     = sum((max(TARGET_H - s + stride - 1, 0) // stride + 1) *
                    (max(TARGET_W - s + stride - 1, 0) // stride + 1) for s in sides)
    done = 0
    for side in sides:
        h_grids = max(TARGET_H - side + stride - 1, 0) // stride + 1
        w_grids = max(TARGET_W - side + stride - 1, 0) // stride + 1
        for i in range(h_grids):
            for j in range(w_grids):
                y1 = i * stride; x1 = j * stride
                y2 = min(y1 + side, TARGET_H); x2 = min(x1 + side, TARGET_W)
                y1 = max(y2 - side, 0);        x1 = max(x2 - side, 0)
                cos = predict_whole_img(img_tensor[:, y1:y2, x1:x2], text_features)
                cos = F.interpolate(cos.unsqueeze(0), size=(y2 - y1, x2 - x1),
                                    mode="bilinear", align_corners=False).squeeze(0)
                probs[:, y1:y2, x1:x2] += cos.softmax(dim=0)
                counts[y1:y2, x1:x2]   += 1
                done += 1
                print(f"  [1/4] Window {done}/{total}  scale={side}px", end="\r")
    print()
    probs /= counts.clamp(min=1)

    print("  [2/4] Raw cosine maps...")
    raw_cos  = predict_whole_img(img_tensor, text_features)
    raw_cos  = F.interpolate(raw_cos.unsqueeze(0), size=(TARGET_H, TARGET_W),
                             mode="bicubic", align_corners=False).squeeze(0)
    heatmaps = raw_cos.detach().cpu().numpy()

    print("  [3/4] Bilateral filter...")
    heatmaps_bf = bilateral_filter_heatmaps(heatmaps, img_np,
                                             d=bilateral_d, sigma_color=bilateral_sc, sigma_space=bilateral_ss)

    pred_idx        = probs.argmax(dim=0).detach().cpu().numpy()
    top2            = torch.topk(probs, k=2, dim=0).values
    margin          = (top2[0] - top2[1]).detach().cpu().numpy()
    uncertain_mask  = margin < margin_threshold
    pred_idx_masked = pred_idx.copy()
    pred_idx_masked[uncertain_mask] = -1

    pred_idx_kmeans = pred_idx.copy()
    cluster_labels  = np.zeros_like(pred_idx)
    if use_kmeans:
        print("  [4/4] K-means refinement...")
        ep      = get_early_patch_features(img_tensor.unsqueeze(0), layer=early_layer)[0].float()
        ep_grid = F.interpolate(
            ep.reshape(N_PATCHES_H, N_PATCHES_W, -1).permute(2, 0, 1).unsqueeze(0),
            size=(TARGET_H, TARGET_W), mode="bilinear", align_corners=False
        ).squeeze(0).permute(1, 2, 0).cpu().numpy()
        pred_idx_kmeans, cluster_labels = kmeans_refine(heatmaps_bf, ep_grid, n_clusters=n_clusters)
    else:
        print("  [4/4] K-means skipped.")

    return heatmaps, pred_idx, pred_idx_masked, pred_idx_kmeans, uncertain_mask, cluster_labels


print("Pipeline functions ready")


In [ ]:
QUERY_COLORS = [
    "#FF3333", "#33FF57", "#3399FF", "#FFD700", "#FF33FF",
    "#00FFFF", "#FF8800", "#AAFFAA", "#FF6699", "#BB88FF",
]

def _hex_rgb(hex_str):
    h = hex_str.lstrip("#")
    return np.array([int(h[i:i+2], 16) / 255.0 for i in (0, 2, 4)])


def visualize_heatmaps(img_pil, texts, heatmaps,
                       pred_idx, pred_idx_masked, uncertain_mask,
                       pred_idx_kmeans=None, title="", save_path=None):
    img_resized = img_pil.resize((TARGET_W, TARGET_H), Image.LANCZOS)
    img_np   = np.array(img_resized) / 255.0
    img_grey = np.stack([img_np.mean(axis=2)] * 3, axis=2)

    n = len(texts)
    assert n <= len(QUERY_COLORS), f"Add more colours to QUERY_COLORS — need {n}, have {len(QUERY_COLORS)}."

    show_kmeans   = pred_idx_kmeans is not None
    header_panels = 4 if show_kmeans else 3
    total_panels  = header_panels + n
    cols = min(total_panels, 4)
    rows = int(np.ceil(total_panels / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), facecolor="#1a1a1a")
    axes = np.array(axes).flatten()
    for ax in axes:
        ax.set_facecolor("#1a1a1a")

    def seg_overlay(pid):
        rgb = np.zeros_like(img_np)
        for i in range(n):
            rgb[pid == i] = _hex_rgb(QUERY_COLORS[i])
        out = 0.4 * img_grey + 0.6 * rgb
        out[pid == -1] = img_np[pid == -1]
        return out

    q_patches = [mpatches.Patch(color=QUERY_COLORS[i], label=texts[i]) for i in range(n)]
    u_patch   = mpatches.Patch(color="#555555", label="uncertain")
    leg_kw    = dict(loc="lower right", fontsize=7, framealpha=0.85, facecolor="#222", labelcolor="white")

    axes[0].imshow(img_np);                axes[0].set_title("Original",       fontsize=11, color="white"); axes[0].axis("off")
    axes[1].imshow(seg_overlay(pred_idx)); axes[1].legend(handles=q_patches, **leg_kw)
    axes[1].set_title("Argmax",            fontsize=11, color="white"); axes[1].axis("off")
    axes[2].imshow(seg_overlay(pred_idx_masked)); axes[2].legend(handles=q_patches + [u_patch], **leg_kw)
    axes[2].set_title(f"Margin-filtered  ({100 * uncertain_mask.mean():.0f}% uncertain)", fontsize=11, color="white"); axes[2].axis("off")

    if show_kmeans:
        axes[3].imshow(seg_overlay(pred_idx_kmeans)); axes[3].legend(handles=q_patches, **leg_kw)
        axes[3].set_title("K-means refined", fontsize=11, color="#FFD700"); axes[3].axis("off")

    _cmap = matplotlib.colormaps["viridis"]
    ALPHA = 0.65
    for i, (text, hm) in enumerate(zip(texts, heatmaps)):
        ax     = axes[header_panels + i]
        lo, hi = hm.min(), hm.max()
        norm   = (hm - lo) / (hi - lo + 1e-8)
        comp   = (img_grey * (1 - ALPHA) + _cmap(norm)[:, :, :3] * ALPHA).clip(0, 1)
        ax.imshow(comp)
        sm = matplotlib.cm.ScalarMappable(cmap="viridis", norm=matplotlib.colors.Normalize(vmin=lo, vmax=hi))
        sm.set_array([])
        cb = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02)
        cb.set_label("cosine similarity", color="white", fontsize=7)
        cb.ax.yaxis.set_tick_params(color="white", labelsize=6)
        plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")
        cb.outline.set_edgecolor("#555555")
        ax.set_title(f'"{text}"', fontsize=10, color="white", fontweight="bold")
        ax.axis("off")

    for j in range(total_panels, len(axes)):
        axes[j].axis("off")
    if title:
        fig.suptitle(title, fontsize=13, color="white", y=1.01)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=120, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Saved -> {save_path}")

    plt.show(); plt.close()


def visualize_query_heatmap(img_pil, query_names, all_query_names,
                             heatmaps_np, title="", save_path=None):
    assert 1 <= len(query_names) <= 2
    img_np   = np.array(img_pil.resize((TARGET_W, TARGET_H), Image.LANCZOS)) / 255.0
    img_grey = np.stack([img_np.mean(axis=2)] * 3, axis=2)
    _cmap    = matplotlib.colormaps["viridis"]
    ALPHA    = 0.70

    fig, axes = plt.subplots(1, len(query_names) + 1,
                              figsize=(6 * (len(query_names) + 1), 5), facecolor="#1a1a1a")
    axes = np.array(axes).flatten()
    for ax in axes:
        ax.set_facecolor("#1a1a1a")

    axes[0].imshow(img_np); axes[0].set_title("Original", fontsize=12, color="white"); axes[0].axis("off")

    for pi, qname in enumerate(query_names):
        ax     = axes[pi + 1]
        q_idx  = all_query_names.index(qname)
        sim    = heatmaps_np[q_idx]
        lo, hi = np.percentile(sim, 1), np.percentile(sim, 99)
        norm   = (sim - lo) / (hi - lo + 1e-8)
        comp   = (img_grey * (1 - ALPHA) + _cmap(norm.clip(0, 1))[:, :, :3] * ALPHA).clip(0, 1)
        ax.imshow(comp)
        sm = matplotlib.cm.ScalarMappable(cmap="viridis", norm=matplotlib.colors.Normalize(vmin=lo, vmax=hi))
        sm.set_array([])
        cb = plt.colorbar(sm, ax=ax, fraction=0.038, pad=0.02)
        cb.set_label("cosine similarity", color="white", fontsize=8)
        cb.ax.yaxis.set_tick_params(color="white", labelsize=7)
        plt.setp(cb.ax.yaxis.get_ticklabels(), color="white")
        cb.outline.set_edgecolor("#555555")
        ax.set_title(f'"{qname}"  (peak sim = {sim.max():.3f})', fontsize=11, color="white", fontweight="bold")
        ax.axis("off")

    if title:
        fig.suptitle(title, fontsize=13, color="white", y=1.01)
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Saved -> {save_path}")

    plt.show(); plt.close()


print("Visualisation functions ready")


## 6b. Pipeline Demo

In [ ]:
demo_texts = ["dog", "bowl", "chair", "wooden floor", "tupperware"]

print("Building text features...")
demo_text_feats = build_text_features(demo_texts)

print("\nRunning full pipeline...")
heatmaps, pred_idx, pred_idx_masked, pred_idx_kmeans, uncertain_mask, cluster_labels = get_heatmaps_full(
    img_pil, demo_text_feats,
    sides=(512, 256), stride=128, early_layer=8,
    bilateral_d=9, bilateral_sc=0.15, bilateral_ss=12,
    margin_threshold=0.08, use_kmeans=True, n_clusters=16,
)

visualize_heatmaps(
    img_pil, demo_texts, heatmaps,
    pred_idx, pred_idx_masked, uncertain_mask,
    pred_idx_kmeans=pred_idx_kmeans,
    title="Demo",
    save_path=os.path.join(OUTPUT_DIR, "example_heatmap.png"),
)


## 7. Dataset Image Resizing

In [ ]:
from pathlib import Path

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
os.makedirs(RESIZED_IMAGES_DIR, exist_ok=True)

source_paths = sorted(p for p in Path(DATASET_IMAGES_DIR).iterdir() if p.suffix in SUPPORTED_EXTS)
print(f"Found {len(source_paths)} images  ->  resizing to {TARGET_W}x{TARGET_H}")

for i, src in enumerate(source_paths):
    dst = Path(RESIZED_IMAGES_DIR) / src.name
    if dst.exists():
        continue
    Image.open(src).convert("RGB").resize((TARGET_W, TARGET_H), Image.LANCZOS).save(dst, quality=95)
    if (i + 1) % 50 == 0 or i == len(source_paths) - 1:
        print(f"  [{i+1}/{len(source_paths)}] {src.name}")

print("Done")


## 8. Batch Inference

In [ ]:
SCENE_QUERIES_INDOOR = [
    "chair", "shelf", "vase", "door",
]
SCENE_QUERIES_OUTDOOR = [
    "dark bronze statue sculpture", "tree and shrub foliage",
    "brick pavement ground", "blue sky", "green grass",
    "concrete building wall", "wooden bench", "stone pedestal base",
]
FOCUS_QUERIES_INDOOR  = ["chair", "wooden door"]
FOCUS_QUERIES_OUTDOOR = ["dark bronze statue sculpture", "brick pavement ground"]

SCENE_QUERIES = SCENE_QUERIES_INDOOR  if SCENE_TYPE == "indoor"  else SCENE_QUERIES_OUTDOOR
FOCUS_QUERIES = FOCUS_QUERIES_INDOOR  if SCENE_TYPE == "indoor"  else FOCUS_QUERIES_OUTDOOR

SLIDE_SIDES   = (512,)
SLIDE_STRIDE  = 256
EARLY_LAYER   = 8
BILATERAL_D   = 9
BILATERAL_SC  = 0.08
BILATERAL_SS  = 12
MARGIN_THRESH = 0.08
USE_KMEANS    = True
N_CLUSTERS    = 48
MAX_IMAGES    = 5

print(f"Scene   : {SCENE_TYPE}")
print(f"Queries : {SCENE_QUERIES}")
print(f"Focus   : {FOCUS_QUERIES}\n")

print("Building text features...")
SCENE_TEXT_FEATS = build_text_features(SCENE_QUERIES)
print(f"Text features: {SCENE_TEXT_FEATS.shape}\n")

resized_paths = sorted(p for p in Path(RESIZED_IMAGES_DIR).iterdir() if p.suffix in SUPPORTED_EXTS)
if MAX_IMAGES is not None:
    resized_paths = resized_paths[:MAX_IMAGES]
print(f"Processing {len(resized_paths)} images...\n")

for img_path in resized_paths:
    print(f"  {img_path.name}")
    img = Image.open(img_path).convert("RGB")

    heatmaps, pred_idx, pred_idx_masked, pred_idx_kmeans, uncertain_mask, cluster_labels = get_heatmaps_full(
        img, SCENE_TEXT_FEATS,
        sides=SLIDE_SIDES, stride=SLIDE_STRIDE, early_layer=EARLY_LAYER,
        bilateral_d=BILATERAL_D, bilateral_sc=BILATERAL_SC, bilateral_ss=BILATERAL_SS,
        margin_threshold=MARGIN_THRESH, use_kmeans=USE_KMEANS, n_clusters=N_CLUSTERS,
    )

    visualize_heatmaps(
        img, SCENE_QUERIES, heatmaps, pred_idx, pred_idx_masked, uncertain_mask,
        pred_idx_kmeans=pred_idx_kmeans,
        title=img_path.name,
        save_path=os.path.join(OUTPUT_DIR, "dataset", img_path.stem + "_heatmap.png"),
    )
    visualize_query_heatmap(
        img, query_names=FOCUS_QUERIES, all_query_names=SCENE_QUERIES,
        heatmaps_np=heatmaps,
        title=f"{img_path.name}",
        save_path=os.path.join(OUTPUT_DIR, "dataset", img_path.stem + "_focused_heatmap.png"),
    )

print("\nDone ->", os.path.join(OUTPUT_DIR, "dataset"))


## 9. Video Export

In [ ]:
import gc
import cv2
import matplotlib

VIDEO_FPS     = 30
VIDEO_QUERIES = FOCUS_QUERIES

PANEL_W = TARGET_W; PANEL_H = TARGET_H
FRAME_W = (PANEL_W * 3) + ((PANEL_W * 3) % 2)
FRAME_H = PANEL_H + (PANEL_H % 2)

VIDEO_DIR  = Path(OUTPUT_DIR) / "video"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_PATH = str(VIDEO_DIR / f"{SCENE_TYPE}_heatmap_{'_'.join(q.replace(' ', '_') for q in VIDEO_QUERIES)}.mp4")

_CMAP         = matplotlib.colormaps["viridis"]
HEATMAP_ALPHA = 0.70

assert all(q in SCENE_QUERIES for q in VIDEO_QUERIES), "VIDEO_QUERIES must be a subset of SCENE_QUERIES"
Q_INDICES = [SCENE_QUERIES.index(q) for q in VIDEO_QUERIES]

all_paths = sorted(p for p in Path(RESIZED_IMAGES_DIR).iterdir() if p.suffix in SUPPORTED_EXTS)
print(f"Frames  : {len(all_paths)}  ->  {len(all_paths)/VIDEO_FPS:.1f}s @ {VIDEO_FPS}fps")
print(f"Output  : {VIDEO_PATH}\n")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_PATH, fourcc, VIDEO_FPS, (FRAME_W, FRAME_H))
if not writer.isOpened():
    raise RuntimeError("VideoWriter failed — check OpenCV has FFMPEG support.")

matplotlib.use("Agg")


def render_frame(img_path):
    img_pil    = Image.open(img_path).convert("RGB")
    img_np     = np.array(img_pil.resize((PANEL_W, PANEL_H), Image.LANCZOS), dtype=np.float32) / 255.0
    img_grey   = np.stack([img_np.mean(axis=2)] * 3, axis=2)

    img_tensor  = load_image(img_pil).squeeze(0)
    raw_cos     = predict_whole_img(img_tensor, SCENE_TEXT_FEATS)
    raw_cos     = F.interpolate(raw_cos.unsqueeze(0), size=(PANEL_H, PANEL_W),
                                mode="bicubic", align_corners=False).squeeze(0)
    heatmaps_np = raw_cos.detach().cpu().numpy()
    del img_tensor, raw_cos
    if DEVICE.type == "mps":
        torch.mps.empty_cache()

    panels = [(img_np * 255).clip(0, 255).astype(np.uint8)]
    for q_idx in Q_INDICES:
        sim    = heatmaps_np[q_idx]
        lo, hi = float(np.percentile(sim, 1)), float(np.percentile(sim, 99))
        norm   = ((sim - lo) / (hi - lo + 1e-8)).clip(0, 1)
        comp   = (img_grey * (1 - HEATMAP_ALPHA) + _CMAP(norm)[:, :, :3] * HEATMAP_ALPHA).clip(0, 1)
        panels.append((comp * 255).astype(np.uint8))
    del heatmaps_np

    safe = []
    for p in panels:
        if p.shape[:2] != (PANEL_H, PANEL_W):
            c = np.zeros((PANEL_H, PANEL_W, 3), dtype=np.uint8)
            c[:min(p.shape[0], PANEL_H), :min(p.shape[1], PANEL_W)] = p[:PANEL_H, :PANEL_W]
            safe.append(c)
        else:
            safe.append(p)

    frame = np.concatenate(safe, axis=1)
    if frame.shape[1] < FRAME_W or frame.shape[0] < FRAME_H:
        c = np.zeros((FRAME_H, FRAME_W, 3), dtype=np.uint8)
        c[:frame.shape[0], :frame.shape[1]] = frame
        frame = c

    for xi, qname in zip([PANEL_W + 10, PANEL_W * 2 + 10], VIDEO_QUERIES):
        cv2.putText(frame, f'"{qname}"', (xi, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2, cv2.LINE_AA)

    return cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)


n_done = n_errors = 0
for img_path in all_paths:
    try:
        frame = render_frame(img_path)
        writer.write(frame)
        del frame
    except Exception as e:
        print(f"  skipped {img_path.name}: {e}")
        n_errors += 1
    finally:
        if DEVICE.type == "mps":
            torch.mps.empty_cache()
        gc.collect()
    n_done += 1
    if n_done % 100 == 0 or n_done == len(all_paths):
        print(f"  [{n_done:>4}/{len(all_paths)}]  {100 * n_done / len(all_paths):.1f}%")

writer.release()
print(f"\n{VIDEO_PATH}  ({n_done} frames, {n_errors} skipped)  {n_done / VIDEO_FPS:.1f}s @ {VIDEO_FPS}fps")
